# MERFISH Epicardial Cell State Visualization

**Dataset:** MERFISH spatial transcriptomics, 12 PCW human fetal heart (https://pmc.ncbi.nlm.nih.gov/articles/PMC12637541/) 

**Author:** Quang Dang  

---

## Overview

This notebook reproduces the spatial co-expression visualizations for epicardial cell states in the 12 PCW human fetal heart MERFISH dataset. Two complementary sectioning planes are used:

- **Coronal section** — anterior (frontal) whole-heart view; used for States 2 and 5 (dual- and triple-marker)
- **Transverse section** — axial (top-down) view; used for State 6 (four-marker framework)

### Visualization approach
All visualizations use a percentile-based, non-parametric contrast normalization (99th percentile upper bound; adjustable lower cutoff). Markers are additively mixed into RGB space, with multi-positive cells rendered as white. Cells are plotted in ascending RGB intensity order so co-expressing cells remain visible.

---

## Table of Contents
1. [Imports and Setup](#1-imports-and-setup)
2. [Helper Functions](#2-helper-functions)
3. [Coronal Section — Dual-Marker Visualization (State 2: WT1 + MKI67)](#3-coronal-dual-marker)
4. [Coronal Section — Triple-Marker Visualization (State 5: WT1 + OSR1 + ENPP2)](#4-coronal-triple-marker)
5. [Transverse Section — Four-Marker Framework (State 6: WT1 + POSTN + EDNRA + ALDH1A2)](#5-transverse-four-marker)

## 1. Imports and Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse
import scanpy as sc

# Display settings
sc.settings.verbosity = 1
plt.rcParams['figure.dpi'] = 150

### Load AnnData objects

Two pre-annotated AnnData objects are required:
- `adata_coronal` — coronal section; must contain `celltype` annotations: `'atrial epicardial'`, `'ventricular epicardial'`
- `adata_transverse` — transverse section; must contain `celltype` annotations: `'Epicardial'`, `'EPDC'`

Cell-type annotations are as provided by the original dataset authors.

In [ ]:
# ── Update paths to match your local environment ──────────────────────────────
CORONAL_PATH    = '/path/to/coronal_section.h5ad'
TRANSVERSE_PATH = '/path/to/transverse_section.h5ad'

adata_coronal    = sc.read(CORONAL_PATH)
adata_transverse = sc.read(TRANSVERSE_PATH)

print('Coronal:   ', adata_coronal)
print('Transverse:', adata_transverse)

---
## 2. Helper Functions

### 2.1 Shared utilities

In [ ]:
def get_expression_vector(adata, gene):
    """
    Retrieve a dense expression vector for a given gene (case-insensitive).
    Returns a 1-D numpy array of length n_obs.
    """
    name = next((x for x in adata.var_names if x.upper() == gene.upper()), None)
    if name is None:
        raise ValueError(f"Gene '{gene}' not found in adata.var_names.")
    idx = adata.var_names.get_loc(name)
    vec = adata.X[:, idx]
    return vec.toarray().flatten() if scipy.sparse.issparse(vec) else vec.flatten()


def normalize_percentile(vec, lower_cutoff_percentile=40, upper_percentile=99):
    """
    Percentile-based, non-parametric contrast normalization.

    Parameters
    ----------
    vec : np.ndarray
        Raw expression vector.
    lower_cutoff_percentile : float
        Percentile used as the lower bound (background subtraction).
        Default: 40th percentile.
    upper_percentile : float
        Percentile used as the upper bound (outlier suppression).
        Default: 99th percentile.

    Returns
    -------
    np.ndarray clipped to [0, 1].
    """
    if np.max(vec) == 0:
        return vec
    vmax = np.percentile(vec, upper_percentile)
    vmin = np.percentile(vec, lower_cutoff_percentile)
    return np.clip((vec - vmin) / (vmax - vmin + 1e-9), 0, 1)

---
<a id="3-coronal-dual-marker"></a>
## 3. Coronal Section — Dual-Marker Visualization (State 2)

Assesses spatial co-expression of **WT1** (epicardial identity) and **MKI67** (proliferation) in atrial and ventricular epicardial cells. Normalized intensities are encoded directly into RGB channels using additive color mixing. Cells exceeding a normalized intensity threshold of **0.2** for both markers are classified as co-expressing and rendered as **white**.

| Gene | Channel | Color |
|------|---------|-------|
| WT1 | Red | Magenta (R+B) |
| MKI67 | Green | Green |

In [ ]:
def plot_merfish_dual_marker(
    adata,
    gene_magenta, gene_green,
    target_groups,
    coexpr_threshold=0.2,
    lower_cutoff=0.40,
    xlim=None, ylim=None,
    spot_size=5,
    flip_x=False, flip_y=False,
    save_path=None, dpi=800
):
    """
    Dual-marker spatial co-expression plot (additive RGB mixing).

    Parameters
    ----------
    adata            : AnnData with .obsm['X_spatial'] and .obs['celltype']
    gene_magenta     : Gene mapped to the magenta channel (R + B)
    gene_green       : Gene mapped to the green channel
    target_groups    : List of celltype labels to display as signal
    coexpr_threshold : Normalized intensity threshold for co-expression (default 0.2)
    lower_cutoff     : Percentile (0–1) subtracted before rescaling (default 0.40 = 40th)
    """
    x = adata.obsm['X_spatial'][:, 0]
    y = adata.obsm['X_spatial'][:, 1]

    v1 = get_expression_vector(adata, gene_magenta)
    v2 = get_expression_vector(adata, gene_green)

    n1 = normalize_percentile(v1, lower_cutoff_percentile=lower_cutoff * 100)
    n2 = normalize_percentile(v2, lower_cutoff_percentile=lower_cutoff * 100)

    # Additive RGB: magenta = R+B, green = G
    colors = np.zeros((adata.n_obs, 3))
    colors[:, 0] = n1  # R
    colors[:, 1] = n2  # G
    colors[:, 2] = n1  # B

    # Co-expressing cells → white
    coexpr_mask = (n1 > coexpr_threshold) & (n2 > coexpr_threshold)
    colors[coexpr_mask] = [1.0, 1.0, 1.0]

    mask_in_group = adata.obs['celltype'].isin(target_groups).values
    colors[~mask_in_group] = 0

    # ── Plotting ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 10))
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    # Ghost layer: all cells as faint background
    ax.scatter(x, y, c='#333333', s=spot_size * 0.5, edgecolors='none')

    # Signal layer: target cells sorted by brightness
    mask_show = mask_in_group & ((n1 > 0) | (n2 > 0))
    if np.any(mask_show):
        intensities = colors[mask_show].sum(axis=1)
        order = np.argsort(intensities)
        ax.scatter(
            x[mask_show][order], y[mask_show][order],
            c=colors[mask_show][order],
            s=spot_size, edgecolors='none'
        )

    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)
    if flip_x: ax.invert_xaxis()
    if flip_y: ax.invert_yaxis()

    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(
        f'{gene_magenta} (magenta)  ·  {gene_green} (green)  ·  co-expr (white)',
        color='white', fontsize=12, pad=8
    )

    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='black')
        print(f'Saved → {save_path}')
    plt.show()

In [ ]:
# ── State 2: WT1 (proliferating epicardium) ───────────────────────────────────
plot_merfish_dual_marker(
    adata_coronal,
    gene_magenta='WT1',
    gene_green='MKI67',
    target_groups=['atrial epicardial', 'ventricular epicardial'],
    coexpr_threshold=0.2,
    lower_cutoff=0.40,
    spot_size=5,
    # save_path='figures/state2_wt1_mki67_coronal.png'
)

---
<a id="4-coronal-triple-marker"></a>
## 4. Coronal Section — Triple-Marker Visualization (State 5)

Assesses spatial co-expression of **WT1**, **OSR1**, and **ENPP2** using structured channel-wise maximum mixing to preserve distinct color identities. Triple-positive cells are rendered as **white**.

| Gene | Channel | Single-positive color |
|------|---------|----------------------|
| WT1 | Magenta (R+B) | Magenta |
| OSR1 | Cyan (G+B) | Cyan |
| ENPP2 | Yellow (R+G) | Yellow |

In [ ]:
def plot_merfish_triple_marker(
    adata,
    gene_magenta, gene_cyan, gene_yellow,
    target_groups,
    coexpr_threshold=0.2,
    lower_cutoff=0.40,
    xlim=None, ylim=None,
    spot_size=5,
    flip_x=False, flip_y=False,
    save_path=None, dpi=800
):
    """
    Triple-marker spatial co-expression plot using structured channel-wise
    maximum mixing (magenta / cyan / yellow encoding).

    Parameters
    ----------
    gene_magenta : Mapped to R+B channels → appears magenta
    gene_cyan    : Mapped to G+B channels → appears cyan
    gene_yellow  : Mapped to R+G channels → appears yellow
    Triple-positive cells are rendered white.
    """
    x = adata.obsm['X_spatial'][:, 0]
    y = adata.obsm['X_spatial'][:, 1]

    v1 = get_expression_vector(adata, gene_magenta)
    v2 = get_expression_vector(adata, gene_cyan)
    v3 = get_expression_vector(adata, gene_yellow)

    n1 = normalize_percentile(v1, lower_cutoff_percentile=lower_cutoff * 100)
    n2 = normalize_percentile(v2, lower_cutoff_percentile=lower_cutoff * 100)
    n3 = normalize_percentile(v3, lower_cutoff_percentile=lower_cutoff * 100)

    # Structured channel-wise maximum mixing
    colors = np.zeros((adata.n_obs, 3))
    colors[:, 0] = np.maximum(n1, n3)  # R = magenta + yellow
    colors[:, 1] = np.maximum(n2, n3)  # G = cyan + yellow
    colors[:, 2] = np.maximum(n1, n2)  # B = magenta + cyan

    # Triple-positive → white
    triple_mask = (n1 > coexpr_threshold) & (n2 > coexpr_threshold) & (n3 > coexpr_threshold)
    colors[triple_mask] = [1.0, 1.0, 1.0]

    mask_in_group = adata.obs['celltype'].isin(target_groups).values
    colors[~mask_in_group] = 0

    # ── Plotting ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 10))
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    ax.scatter(x, y, c='#333333', s=spot_size * 0.5, edgecolors='none')

    mask_show = mask_in_group & ((n1 > 0) | (n2 > 0) | (n3 > 0))
    if np.any(mask_show):
        intensities = colors[mask_show].sum(axis=1)
        order = np.argsort(intensities)
        ax.scatter(
            x[mask_show][order], y[mask_show][order],
            c=colors[mask_show][order],
            s=spot_size, edgecolors='none'
        )

    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)
    if flip_x: ax.invert_xaxis()
    if flip_y: ax.invert_yaxis()

    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(
        f'{gene_magenta} (magenta)  ·  {gene_cyan} (cyan)  ·  {gene_yellow} (yellow)  ·  triple+ (white)',
        color='white', fontsize=11, pad=8
    )

    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='black')
        print(f'Saved → {save_path}')
    plt.show()

In [ ]:
# ── State 5: sub-epicardial / mesenchymal state ───────────────────────────────
plot_merfish_triple_marker(
    adata_coronal,
    gene_magenta='WT1',
    gene_cyan='OSR1',
    gene_yellow='ENPP2',
    target_groups=['atrial epicardial', 'ventricular epicardial'],
    coexpr_threshold=0.2,
    lower_cutoff=0.40,
    spot_size=5,
    # save_path='figures/state5_wt1_osr1_enpp2_coronal.png'
)

---
<a id="5-transverse-four-marker"></a>
## 5. Transverse Section — Four-Marker Framework (State 6)

Spatially resolves the **migratory epicardial cell state (State 6)** using four markers. Three markers (WT1, POSTN, EDNRA) are mixed into RGB space using structured channel-wise maximum mixing. The fourth marker (ALDH1A2) acts as a gating feature: cells must exceed a normalized threshold of **0.2** for all four markers to be rendered white. An exclusion step removes cells expressing **ITLN1** above threshold (State 4 contamination).

| Gene | Role | Channel |
|------|------|---------|
| WT1 | Epicardial identity | Magenta (R+B) |
| POSTN | Migratory ECM | Cyan (G+B) |
| EDNRA | Migratory receptor | Yellow (R+G) |
| ALDH1A2 | Gating marker | — (white gate) |
| ITLN1 | Exclusion marker | — (masks State 4) |

In [ ]:
def plot_merfish_quad_marker(
    adata,
    gene_magenta, gene_cyan, gene_yellow, gene_gate,
    target_groups,
    coexpr_threshold=0.2,
    lower_cutoff=0.40,
    exclude_genes=None,
    exclude_threshold=0.2,
    xlim=None, ylim=None,
    spot_size=5,
    flip_x=False, flip_y=False,
    save_path=None, dpi=800
):
    """
    Four-marker spatial co-expression plot for the transverse MERFISH dataset.

    Three RGB markers (magenta / cyan / yellow) are mixed using structured
    channel-wise maximum mixing. A fourth marker gates quad-positive cells
    (rendered white). Optional exclusion genes remove confounding populations.

    Parameters
    ----------
    adata           : AnnData with .obsm['X_spatial'] and .obs['celltype']
    gene_magenta    : Mapped to R+B channels
    gene_cyan       : Mapped to G+B channels
    gene_yellow     : Mapped to R+G channels
    gene_gate       : Fourth marker; gates quad-positive (white) cells
    target_groups   : celltype labels to include as signal
    coexpr_threshold: Normalized intensity threshold for quad-positive gate
    lower_cutoff    : Percentile (0–1) for lower normalization bound
    exclude_genes   : List of genes; cells exceeding exclude_threshold are masked
    exclude_threshold: Normalized intensity cutoff for exclusion genes
    """
    x = adata.obsm['X_spatial'][:, 0]
    y = adata.obsm['X_spatial'][:, 1]

    v1 = get_expression_vector(adata, gene_magenta)
    v2 = get_expression_vector(adata, gene_cyan)
    v3 = get_expression_vector(adata, gene_yellow)
    v4 = get_expression_vector(adata, gene_gate)

    q = lower_cutoff * 100
    n1 = normalize_percentile(v1, lower_cutoff_percentile=q)
    n2 = normalize_percentile(v2, lower_cutoff_percentile=q)
    n3 = normalize_percentile(v3, lower_cutoff_percentile=q)
    n4 = normalize_percentile(v4, lower_cutoff_percentile=q)

    # ── Exclusion mask ────────────────────────────────────────────────────────
    mask_exclude = np.zeros(adata.n_obs, dtype=bool)
    if exclude_genes:
        for g in exclude_genes:
            try:
                ve = get_expression_vector(adata, g)
                ne = normalize_percentile(ve, lower_cutoff_percentile=q)
                mask_exclude |= (ne > exclude_threshold)
            except ValueError as e:
                print(f'Warning: {e}')

    # ── Structured channel-wise maximum mixing ────────────────────────────────
    colors = np.zeros((adata.n_obs, 3))
    colors[:, 0] = np.maximum(n1, n3)  # R
    colors[:, 1] = np.maximum(n2, n3)  # G
    colors[:, 2] = np.maximum(n1, n2)  # B

    # Quad-positive → white
    t = coexpr_threshold
    quad_mask = (n1 > t) & (n2 > t) & (n3 > t) & (n4 > t)
    colors[quad_mask] = [1.0, 1.0, 1.0]

    # Apply group membership and exclusion
    mask_in_group = adata.obs['celltype'].isin(target_groups).values
    valid_mask = mask_in_group & (~mask_exclude)
    colors[~valid_mask] = 0

    # ── Plotting ──────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 10))
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    # Ghost layer
    ax.scatter(x, y, c='#333333', s=spot_size * 0.5, edgecolors='none')

    # Signal layer
    mask_show = valid_mask & ((n1 > 0) | (n2 > 0) | (n3 > 0) | (n4 > 0))
    if np.any(mask_show):
        intensities = colors[mask_show].sum(axis=1)
        order = np.argsort(intensities)
        ax.scatter(
            x[mask_show][order], y[mask_show][order],
            c=colors[mask_show][order],
            s=spot_size, edgecolors='none'
        )

    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)
    if flip_x: ax.invert_xaxis()
    if flip_y: ax.invert_yaxis()

    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(
        f'{gene_magenta} (M)  ·  {gene_cyan} (C)  ·  {gene_yellow} (Y)  ·  gated by {gene_gate}  ·  quad+ (white)',
        color='white', fontsize=10, pad=8
    )

    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight', facecolor='black')
        print(f'Saved → {save_path}')
    plt.show()

In [ ]:
# ── State 6: migratory epicardial cells — atrioventricular junction region ────
plot_merfish_quad_marker(
    adata_transverse,
    gene_magenta='WT1',
    gene_cyan='POSTN',
    gene_yellow='EDNRA',
    gene_gate='ALDH1A2',
    target_groups=['Epicardial', 'EPDC'],
    exclude_genes=['ITLN1'],
    coexpr_threshold=0.2,
    lower_cutoff=0.40,
    xlim=(12000, 18500),
    ylim=(-13500, -8000),
    spot_size=5,
    flip_y=False,
    flip_x=False,
    # save_path='figures/state6_wt1_postn_ednra_aldh1a2_transverse_AVJ.png'
)

---
## Session Info

In [ ]:
import session_info
session_info.show()